# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 2: KỸ THUẬT ĐẶC TRƯNG (FEATURE ENGINEERING) & THIẾT LẬP DÂY CHUYỀN XỬ LÍ (PIPELINE)

> **Mục tiêu chính:** 
> - Tạo thêm các thước đo đặc trưng từ nội dung email (viết hoa, dấu cảm thán, tiền tệ, tiền tố trả lời).
> - Chia dữ liệu làm 2 phần: 80% (Train) và 20% (Test).
> - Đóng gói toàn bộ các bước biến đổi dữ liệu thành một dây chuyền tự động (`Pipeline`).

---
## Khai báo Thư viện & Nạp Dữ liệu Sạch từ Notebook 1

In [ ]:
# Khai báo thư viện & Nạp dữ liệu sạch từ Notebook 1
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

try:
    from IPython.display import display
except ImportError:
    pass

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

clean_path = os.path.join('..', 'data', 'processed', 'clean_emails.csv')
if not os.path.exists(clean_path):
    clean_path = os.path.join('data', 'processed', 'clean_emails.csv')

df = pd.read_csv(clean_path)
print(f"Nạp thành công: {df.shape[0]:,} dòng, {df.shape[1]} cột")

---
## Mục 4: Kỹ thuật Trích xuất Đặc trưng Miền & Biến đổi (Domain Feature Engineering)

### 4.1 Trích xuất Đặc trưng Miền & Phân tích Phân phối (Domain Feature Extraction & Distribution Analysis)
**Mục tiêu Kỹ thuật:**
- Dựa trên các phát hiện từ quá trình (EDA) ở Notebook 1 nên sẽ tạo thêm 7 đặc trưng số giúp mô hình phân biệt chính xác hơn:
  1. `text_len`: Chiều dài ký tự của chuỗi văn bản sạch `text_clean`.
  2. `log_text_len`: Biến đổi logarit $\ln(1 + x)$ nhằm nén phân phối lệch phải cực đoan của độ dài văn bản.
  3. `caps_ratio`: Tỷ lệ chữ IN HOA trên toàn bộ email (thư rác thường viết hoa nhiều để gây chú ý)
  4. `exclamation_count`: Tần suất xuất hiện dấu chấm than (`!`) nhằm đo lường mức độ giật gân, cấp bách.
  5. `dollar_count`: Tần suất ký hiệu tài chính (`$`) thể hiện lời mời chào tiền thưởng hoặc giao dịch đáng ngờ.
  6. `has_re_prefix`: Email có tiền tố trả lời `Re:` ở đầu tiêu đề hay không (1: Có, 0: Không)
  7. `subject_exclamation`: Tiêu đề có chứa dấu `!` hay không (1: Có, 0: Không)
- Rời rạc hóa (`pd.qcut`) độ dài văn bản thành 4 khoảng tứ phân vị phục vụ phân tích nhóm.
- Tính toán và hiển thị bảng thống kê phân phối (mean, median, std) theo nhãn (Ham vs Spam) của các đặc trưng miền để kiểm chứng tính phân biệt trước khi đưa vào Pipeline.

In [ ]:
# 1. Trích xuất đặc trưng độ dài và biến đổi phi tuyến
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['text_len'] = df['text_clean'].fillna('').astype(str).str.len()
df['log_text_len'] = np.log1p(df['text_len'])
df['length_bin'] = pd.qcut(df['text_len'], q=4, labels=['Ngắn', 'Trung bình', 'Dài', 'Rất dài'])

# 2. Trích xuất đặc trưng miền ngữ cảnh từ văn bản gốc (subject + body)
raw_text = df['subject'].fillna('').astype(str) + ' ' + df['body'].fillna('').astype(str)
caps = raw_text.apply(lambda s: sum(1 for c in s if c.isupper()))
total = raw_text.apply(len).replace(0, 1)
df['caps_ratio'] = caps / total
df['exclamation_count'] = raw_text.apply(lambda s: s.count('!'))
df['dollar_count'] = raw_text.apply(lambda s: s.count('$'))

# 3. Bổ sung 2 đặc trưng từ tiêu đề phát hiện trong EDA
df['has_re_prefix'] = df['subject'].fillna('').astype(str).str.contains(r'^\s*re:', case=False, regex=True).astype(int)
df['subject_exclamation'] = df['subject'].fillna('').astype(str).str.contains(r'!', regex=False).astype(int)

# 4. Hiển thị mẫu dữ liệu đặc trưng trích xuất
feature_preview = ['log_text_len', 'caps_ratio', 'exclamation_count', 'dollar_count', 'has_re_prefix', 'subject_exclamation', 'label']
print("Mẫu 5 dòng đặc trưng miền trích xuất:")
display(df[feature_preview].head())

# 5. Bảng thống kê phân phối (mean, median, std) theo nhãn Ham (0) vs Spam (1)
domain_stats_cols = ['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'log_text_len', 'has_re_prefix', 'subject_exclamation']
domain_distribution = df.groupby('label')[domain_stats_cols].agg(['mean', 'median', 'std']).round(4).T
print("\nBảng thống kê phân phối đặc trưng miền theo nhãn (Ham vs Spam):")
display(domain_distribution)

**Đúc kết & Định hướng:**
- **Tính phân biệt vượt trội của tiền tố tiêu đề:** Đặc trưng `has_re_prefix` mang tính phân biệt rất cao: xuất hiện ở 55.26% thư Ham nhưng chỉ có 0.75% ở thư Spam, phản ánh bản chất luồng đàm thoại công việc thực tế.
- **Tín hiệu cảnh báo giật gân:** Thư Spam có tỷ lệ xuất hiện dấu chấm than trên tiêu đề cao gấp 5 lần so với Ham (11.06% so với 2.17%).
- **Chữ in hoa và ký hiệu tiền:** Phân bố chữ viết hoa và ký tự `$` giữa hai nhóm có sự khác biệt rõ rệt.
- **Biến đổi logarit:** Hàm log thu hẹp khoảng cách giữa các thư siêu dài, giúp bước chuẩn hóa số học `MinMaxScaler` phía sau hoạt động ổn định.

---
## Mục 5: Phân chia Dữ liệu & Lắp ráp dây chuyền xử lý (Pipeline)

> Kết hợp chuỗi văn bản sạch `text_clean` cùng 6 đặc trưng số học vào một khối tiền xử lý tự động chung (`ColumnTransformer`) để đưa vào mô hình học máy

### 5.1 Phân chia Train & Test
**Mục tiêu Kỹ thuật:**
- Chia dữ liệu theo tỷ lệ giữa tập Train (80%) và tập Test (20%).
- Áp dụng phân tầng theo nhãn (`stratify=y`) với `RANDOM_SEED = 42` nhằm bảo toàn tuyệt đối tỷ lệ phân bố giữa hai lớp, ngăn ngừa rò rỉ dữ liệu (Data Leakage) sang tập kiểm thử.
- Bộ đặc trưng đầu vào gồm: `text_clean` và 6 biến: `['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'has_re_prefix', 'subject_exclamation']`.

In [ ]:
# Phân chia Train/Test 80/20 phân tầng chống rò rỉ dữ liệu
feature_cols = ['text_clean', 'caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'has_re_prefix', 'subject_exclamation']
X = df[feature_cols]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
print(f"Train: {len(X_train):,} mẫu (Tỷ lệ Spam: {y_train.mean()*100:.2f}%)")
print(f"Test:  {len(X_test):,} mẫu (Tỷ lệ Spam: {y_test.mean()*100:.2f}%)")

**Kết quả chia tập dữ liệu:**
- Đã chia tách độc lập: 26.128 mẫu tập Train và 6.532 mẫu tập Test.
- Tỷ lệ thư rác ở cả hai tập được bảo toàn chuẩn xác ở mức 47,66%.
- Mọi thao tác học từ vựng (TF-IDF), chấm điểm (Chi-Square) và co giãn số học (MinMaxScaler) chỉ được học trên tập Train rồi mới chuyển đổi sang tập Test.

### 5.2 Xây dựng dây chuyền xử lý (Pipline) tự động (`ColumnTransformer`)
Dây chuyền gồm 2 nhánh xử lý chuyên biệt chạy song song:
1. **Nhánh văn bản (`text_clean`):** Dùng `TfidfVectorizer` trích xuất từ đơn và cụm 2 từ (`ngram_range=(1, 2)`), sau đó dùng thuật toán `SelectKBest(chi2)` lọc lấy đúng 1.500 từ có khả năng phân biệt thư rác tốt nhất.
2. **Nhánh số học (`num_cols`):** Dùng `MinMaxScaler` đưa 6 số đo về chung thang điểm $[0, 1]$ để các số lớn không lấn át số nhỏ và tránh giá trị âm gây lỗi cho Naive Bayes.

In [ ]:
# Xây dựng Pipeline Tiền xử lý Đa tầng kết hợp Văn bản và Thuộc tính Số học
num_cols = ['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'has_re_prefix', 'subject_exclamation']

text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3, stop_words='english')),
    ('select', SelectKBest(score_func=chi2, k=1500))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_pipeline, 'text_clean'),
        ('num', MinMaxScaler(), num_cols)
    ]
)

**Kết quả thiết lập Pipeline:**
- Pipeline đa tầng kết hợp thành công nhánh văn bản cùng nhánh số học tạo ra không gian biểu diễn gồm đúng 1.506 đặc trưng (1.500 từ vựng quan trọng + 6 đặc trưng số).
- Khối `ColumnTransformer` đóng vai trò kiến trúc chuẩn hóa đầu vào, bảo đảm bảo đảm dữ liệu mới đưa vào đều được biến đổi đồng nhất

### 5.3 Lưu trữ dữ liệu và dây chuyền tiền xử lý
- Lưu tập dữ liệu đã chia (`X_train`, `X_test`, `y_train`, `y_test`) vào tệp `data/processed/train_test_data.joblib`.
- Lưu dây chuyền xử lý (`preprocessor`) vào tệp `models/preprocessor.joblib`.
- Bàn giao đầy đủ tài nguyên đã chuẩn hóa cho Notebook 3 phục vụ công đoạn huấn luyện, đánh giá chéo phân tầng và tinh chỉnh siêu tham số.

In [ ]:
# Lưu trữ dữ liệu và preprocessor
data_dir = os.path.join('..', 'data', 'processed') if os.path.exists(os.path.join('..', 'data', 'processed')) else os.path.join('data', 'processed')
models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else os.path.join('models')

os.makedirs(data_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

joblib.dump({
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}, os.path.join(data_dir, 'train_test_data.joblib'))

joblib.dump(preprocessor, os.path.join(models_dir, 'preprocessor.joblib'))
print("Đã lưu thành công train_test_data.joblib và preprocessor.joblib.")

**Tổng kết Giai đoạn 2 :**
- Hoàn thành trích xuất 7 đặc trưng miền nghiệp vụ và chia tập Train/Test 80/20 phân tầng.
- Dây chuyền xử lý kết hợp chữ và số (1.506 đặc trưng) đã được đóng gói an toàn thành các tệp `.joblib`.